## Goal: Finalize analysis to answer our 3 supporting questions
- Do fires cross wetland boundaries? (yes, supported by fact that there is wetland area burned each year)
    - 
- Are wildfires less frequent near wetlands? 
    - Goal: use chi squared test to see if burned area inside wetlands is about what you would expect for average of all burned area or if it is statistically different

## Set up all the data

In [3]:
# import packages

import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris
from shapely.geometry import Point
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact


In [4]:
# import data 

# filepath

project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# #  fire data by year
fire_2020 = gpd.read_file(processed_fldr / "fire_explore/fire_2020.geojson")
fire_2021 = gpd.read_file(processed_fldr / "fire_explore/fire_2021.geojson")
fire_2022 = gpd.read_file(processed_fldr / "fire_explore/fire_2022.geojson")
fire_2023 = gpd.read_file(processed_fldr / "fire_explore/fire_2023.geojson")
fire_2024 = gpd.read_file(processed_fldr / "fire_explore/fire_2024.geojson")
fire_2025 = gpd.read_file(processed_fldr / "fire_explore/fire_2025.geojson")

# california outline
ca_outline = gpd.read_file(processed_fldr /"fire_explore/ca_outline.geojson")


# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")



Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [5]:
# convert CRS function
def crs_convert(shp):
    shp = shp.to_crs(epsg=26910)
    return shp

# check CRS of files
print(fire_2020.crs)
print(ca_outline.crs)

EPSG:4326
EPSG:4326


In [17]:
wetlands.head()

,C Code,NWI Type,C System,C Subsystem,C Class,Hectares,NWI ID,geometry
0,R4SBC,Riverine,Riverine,Intermittent,Streambed,3.801841,202409CSw{B818AEF7-DD4D-43B6-A532-1DC94D248F88...,"POLYGON ((760230.569 3758978.374, 760229.722 3..."
1,R4SBC,Riverine,Riverine,Intermittent,Streambed,1.970544,202409CSw{80E28DA5-6244-45AB-9DC9-774ACABD2846...,"POLYGON ((756628.261 3764081.939, 756627.37 37..."
2,R4SBC,Riverine,Riverine,Intermittent,Streambed,2.980342,202409CSw{A66FA93B-E12A-488A-9891-E7526D1AACCE...,"POLYGON ((759665.963 3766263.978, 759666.614 3..."
3,R4SBC,Riverine,Riverine,Intermittent,Streambed,1.254393,202409CSw{B5A926CC-AF98-487F-A4B9-A4789311100A...,"POLYGON ((759095.322 3766226.414, 759096.989 3..."
4,R4SBC,Riverine,Riverine,Intermittent,Streambed,8.104766,202409CSw{C99C89FF-6622-43DE-A93F-138503071611...,"POLYGON ((758401.072 3766310.385, 758403.257 3..."


In [6]:
# convert all shp CRS
fire_2020 = crs_convert(fire_2020)
fire_2021 = crs_convert(fire_2021)
fire_2022 = crs_convert(fire_2022)
fire_2023 = crs_convert(fire_2023)
fire_2024 = crs_convert(fire_2024)
fire_2025 = crs_convert(fire_2025)
ca_outline = crs_convert(ca_outline)
wetlands = crs_convert(wetlands)

In [36]:
# Generate 500,000random points
n_pts = 500000

# Extract California polygon
study_area = ca_outline.geometry.iloc[0]

# Generate random points
random_pts = gpd.GeoSeries(
    [study_area],
    crs=ca_outline.crs
).sample_points( # take a random sample
    size=n_pts,
    method="uniform",
    seed=42 # but get the same random pts every time i run this code
).explode(index_parts=False).reset_index(drop=True) # separate each pt into its own line

# Convert to GeoDataFrame
random_pts = gpd.GeoDataFrame(
    geometry=random_pts,
    crs=ca_outline.crs
)


In [37]:
# place fires in dict for easy access
fires = {
    2020: fire_2020,
    2021: fire_2021,
    2022: fire_2022,
    2023: fire_2023,
    2024: fire_2024,
    2025: fire_2025
}

## Use random points to determine correlation between wetlands and fire
- Question: is land inside wetlands less likely to burn?
- Results appear not statistically significant, so disregard this analysis for now

In [38]:
# function for chi squared analysis, Y/N wetlands and Y/N fire
def classify_pts(fire):

    # create copy of random points
    points = random_pts.copy().reset_index(drop=True)

    # create a copy of wetlands
    wetlands_cpy = wetlands.copy().reset_index(drop=True)

    # identify wetland intersections
    wetland_join = gpd.sjoin(
        points,
        wetlands_cpy,
        how='left'
    )

    # identify C System for each point
    points["C System"] = (
        wetland_join["C System"]
        .groupby(level=0)
        .first()
    )

    # classify wetland presence (1=Y, 0=N)
    points["wetland"] = (
        wetland_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    # identify fire intersections
    fire_join = gpd.sjoin(
        points,
        fire,
        how='left'
    )

    # classify fire presence
    points["fire"] = (
        fire_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    return points

In [39]:
# run function on each year of shapefile
results = {}
for year, fire in fires.items():
    results[year] = classify_pts(fire)

In [40]:
# create contingency tables by year and C System

tables = {}

for year, points in results.items():

    # identify non-wetland points
    non_wetlands = points[points["C System"].isna()]

    # identify each wetland system
    for system in points["C System"].dropna().unique():

        wetlands_system = points[points["C System"] == system]

        # combine wetland system with non-wetland points
        comparison = pd.concat([
            wetlands_system,
            non_wetlands
        ])

        # create contingency table
        tables[(year, system)] = pd.crosstab(
            comparison["wetland"],
            comparison["fire"]
        )

In [42]:
# use chi squared analysis to determine if there is any statistical relationship
# null hypothesis: there is no relationship between wetland occurrence and fire occurrence

# store Chi-squared test results
chi2_results = {}

for (year, system), table in tables.items():

    # run Chi-squared test
    chi2_stat, p_value, dof, expected = chi2_contingency(table)

    # store results
    chi2_results[(year, system)] = {
        "Chi-squared Statistic": chi2_stat,
        "P-value": p_value,
        "Degrees of Freedom": dof,
        "Significant": p_value < 0.05
    }

In [43]:
# convert chi squared results to df
chi2_df = pd.DataFrame.from_dict(
    chi2_results,
    orient="index"
)

chi2_df.index = pd.MultiIndex.from_tuples(
    chi2_df.index,
    names=["Year", "C System"]
)

print(chi2_df)

                 Chi-squared Statistic       P-value  Degrees of Freedom  \
Year C System                                                              
2020 Lacustrine             343.525345  1.089242e-76                   1   
     Palustrine             207.041088  6.073483e-47                   1   
     Riverine                20.237228  6.840866e-06                   1   
     Estuarine               37.428828  9.481034e-10                   1   
     Marine                   3.298784  6.933119e-02                   1   
2021 Lacustrine             192.629157  8.481313e-44                   1   
     Palustrine              28.861834  7.772952e-08                   1   
     Riverine                33.144321  8.556590e-09                   1   
     Estuarine               21.499357  3.539474e-06                   1   
     Marine                   1.583723  2.082252e-01                   1   
2022 Lacustrine              23.479567  1.262475e-06                   1   
     Palustr

In [45]:
# save df in case so it does not run forever

chi2_df.to_csv(processed_fldr/"chi2.csv", index=False)

In [46]:
# use fischer's exact test to determine if there is any statistical relationship 
# null hypothesis: there is no relationship between wetland occurance and fire occurance

# test for 2020 
odds_ratio, p_fisher = fisher_exact(tables[2020])

print("Odds ratio:", odds_ratio) # odds ratio below 1 = wetland locations had lower observed odds of burning than non-wetland locations.
print("Fisher's exact p-value:", p_fisher) # p value below 0.05 indicates statistical significance

KeyError: 2020

In [ ]:
# run fischer's exact test for every year of data
# Odds ratio below 1 = non-wetland locations had lower observed odds of burning than wetland locations.
# Store Fisher's exact test results
fisher_results = {}

for (year, system), table in tables.items():

    # Run Fisher's exact test
    odds_ratio, p_value = fisher_exact(table)

    # Store results
    fisher_results[(year, system)] = {
        "Wetland Type": system,
        "Year": year,
        "Odds Ratio": odds_ratio,
        "P-value": p_value,
        "Significant": p_value < 0.05
    }

In [ ]:
# Convert results into a dataframe


# Odds ratio	Interpretation
# OR < 1	Non-wetland locations had lower odds of burning than wetland locations.
# OR = 1	Wetland and non-wetland locations had equal odds of burning.
# OR > 1	Non-wetland locations had higher odds of burning than wetland locations.
# OR = 0	No sampled wetland locations intersected fire, assuming the other cells are nonzero.
# OR = infinity	No sampled non-wetland locations intersected fire, assuming the other cells are nonzero.

fisher_df = pd.DataFrame.from_dict(
    fisher_results,
    orient="index"
)

# Reset index
fisher_df = fisher_df.reset_index(drop=True)

# Display results
print(fisher_df)

   Wetland Type  Year  Odds Ratio        P-value  Significant
0    Lacustrine  2020    0.080508  2.038615e-124         True
1    Palustrine  2020    0.215900   2.108625e-66         True
2      Riverine  2020    0.678495   2.228236e-06         True
3     Estuarine  2020    0.000000   8.317941e-17         True
4        Marine  2020    0.000000   3.689908e-02         True
5    Lacustrine  2021    0.099942   1.560033e-69         True
6    Palustrine  2021    0.611935   9.531315e-09         True
7      Riverine  2021    0.471988   1.770356e-10         True
8     Estuarine  2021    0.000000   3.703689e-10         True
9        Marine  2021    0.000000   1.828660e-01        False
10   Lacustrine  2022    0.099509   1.109229e-09         True
11   Palustrine  2022    1.385896   7.016635e-02        False
12     Riverine  2022    1.159643   5.213509e-01        False
13    Estuarine  2022    0.000000   1.251741e-01        False
14       Marine  2022    0.000000   1.000000e+00        False
15   Lac

In [49]:
# save fischers df since it takes a long time to run
fisher_df.to_csv(processed_fldr/"fisher.csv", index=False)

## Use random points to help determine if land near wetlands is less likely to burn
- Is land closer to wetlands less likely to burn?

In [ ]:

import time

# goal: generate random points, and classify their distance to a wetland and presence of fire (t/f)
# to answer Q "is land closer to wetlands less likely to burn?"

def classify_dist(fire):

    total_start = time.perf_counter()

    # create copy of random points
    step_start = time.perf_counter()
    points = random_pts.copy().reset_index(drop=True)
    print(f"Copy random points: {time.perf_counter() - step_start:.2f} seconds")

    # create a copy of wetlands
    step_start = time.perf_counter()
    wetlands_cpy = wetlands.copy().reset_index(drop=True)
    print(f"Copy wetlands: {time.perf_counter() - step_start:.2f} seconds")

    # calculate distance to nearest wetland
    step_start = time.perf_counter()
    nearest_wetlands = gpd.sjoin_nearest(
        points,
        wetlands_cpy,
        how="left",
        distance_col="wetland_distance"
    )

    # keep one distance per random point (including tied nearest wetlands)
    nearest_distances = (
        nearest_wetlands
        .groupby(level=0)["wetland_distance"]
        .min()
    )

    points["wetland_distance"] = nearest_distances.reindex(points.index)

    print(f"Calculate nearest wetland distances: {time.perf_counter() - step_start:.2f} seconds")

    # classify distance from wetlands
    step_start = time.perf_counter()
    points["distance_category"] = pd.cut(
        points["wetland_distance"],
        bins=[-1, 1000, 5000, 10000, 20000, float("inf")],
        labels=["0-1 km", "1-5 km", "5-10 km",
                "10-20 km", "Over 20 km"]
    )
    print(f"Classify distances: {time.perf_counter() - step_start:.2f} seconds")

    # identify wetland intersections
    step_start = time.perf_counter()
    wetland_join = gpd.sjoin(
        points,
        wetlands_cpy,
        how="left",
        predicate="intersects"
    )
    print(f"Wetland spatial join: {time.perf_counter() - step_start:.2f} seconds")

    # classify wetland presence (1=Y, 0=N)
    points["wetland"] = (
        wetland_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    # identify fire intersections
    step_start = time.perf_counter()
    fire_join = gpd.sjoin(
        points,
        fire,
        how="left",
        predicate="intersects"
    )
    print(f"Fire spatial join: {time.perf_counter() - step_start:.2f} seconds")

    # classify fire presence
    points["fire"] = (
        fire_join["index_right"]
        .notna()
        .groupby(level=0)
        .any()
        .astype(int)
    )

    print(f"TOTAL FUNCTION TIME: {time.perf_counter() - total_start:.2f} seconds")

    return points



In [ ]:
# test our distance function
analysis_2020 = classify_dist(fire_2020)

print(analysis_2020[
    ["wetland_distance", "distance_category", "fire"]
].head(10))

Copy random points: 0.00 seconds
Copy wetlands: 0.02 seconds
